In [1]:
!pip install -q unsloth[colab-new]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.5 MB/s eta 0:00:00
   ━

In [2]:
!pip install  --no-deps xformers trl peft accelerate bitsandbytes

In [3]:
from unsloth import FastLanguageModel
import torch

# Load the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Inject the LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [4]:
# Load the Spider Dataset
from datasets import load_dataset

# Spider is the gold standard for text-to-SQL tasks
dataset = load_dataset("spider", split="train")

def format_prompt(example):
    # This prepares the data: Question + Schema -> SQL
    return f"Question: {example['question']}\nSchema: {example['db_id']}\nSQL: {example['query']}"

# Apply the formatting to the dataset
formatted_dataset = dataset.map(lambda x: {"text": format_prompt(x)})

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [5]:
# Cell: Inspect Data
print("Dataset type:", type(formatted_dataset))
print("First item keys:", formatted_dataset[0].keys())
print("First item content (truncated):", str(formatted_dataset[0]['text'])[:200])

Dataset type: <class 'datasets.arrow_dataset.Dataset'>
First item keys: dict_keys(['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'text'])
First item content (truncated): Question: How many heads of the departments are older than 56 ?
Schema: department_management
SQL: SELECT count(*) FROM head WHERE age  >  56


In [6]:
# Keep only the 'text' column, remove everything else
formatted_dataset = formatted_dataset.select_columns(["text"])
print("New columns:", formatted_dataset.column_names)

New columns: ['text']


In [7]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=2048)

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

# Now, set the format to PyTorch tensors
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [9]:
# Tokenize and set labels
def tokenize_function(examples):
    # Tokenize the text
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=2048)
    
    # In language modeling, the labels are the input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [11]:
# 1. Re-load and re-format the dataset to its raw state
formatted_dataset = dataset.map(lambda x: {"text": format_prompt(x)})

# 2. Select only the "text" column
formatted_dataset = formatted_dataset.select_columns(["text"])

# 3. Inspect to be sure it's back to the simple format
print("Dataset columns after reset:", formatted_dataset.column_names)
print("Example content:", formatted_dataset[0])

Dataset columns after reset: ['text']
Example content: {'text': 'Question: How many heads of the departments are older than 56 ?\nSchema: department_management\nSQL: SELECT count(*) FROM head WHERE age  >  56'}


In [13]:
# Training the model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 500,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        remove_unused_columns = False, # Keep this False for SFTTrainer
        average_tokens_across_devices = False, # <--- THIS FIXES THE BUG
    ),
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss
1,2.529008
2,2.576275
3,2.941271
4,2.424365
5,2.573532
6,2.400579
7,2.562153
8,2.138167
9,2.012626
10,1.893889


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.4992128352324168, metrics={'train_runtime': 227.5891, 'train_samples_per_second': 4.218, 'train_steps_per_second': 0.264, 'total_flos': 3778706029117440.0, 'train_loss': 1.4992128352324168, 'epoch': 0.13714285714285715})

In [14]:
# 1. Put the model into "Prediction Mode" (Inference)
# This speeds up the model since it doesn't need to calculate loss or gradients anymore
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 2. Pick a test question and schema (Let's use one it might have seen, or make one up!)
test_question = "How many departments are managed by heads older than 56?"
test_schema = "department_management"

# 3. Format the prompt EXACTLY how we trained it, but leave the SQL part blank!
test_prompt = f"Question: {test_question}\nSchema: {test_schema}\nSQL: "

print("Feeding this to the AI:\n" + test_prompt + "\n")

# 4. Tokenize the prompt and send it to the GPU
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# 5. Generate the answer! (max_new_tokens limits how long the SQL can be)
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

# 6. Decode the numbers back into readable text
result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("--- AI PREDICTION ---")
print(result)

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Feeding this to the AI:
Question: How many departments are managed by heads older than 56?
Schema: department_management
SQL: 



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

--- AI PREDICTION ---
Question: How many departments are managed by heads older than 56?
Schema: department_management
SQL:  SELECT count(*) FROM Department WHERE head_age  >  56;


In [30]:
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Put the model into high-speed prediction mode
FastLanguageModel.for_inference(model)

# 2. Grab the first 1000 real questions from your Spider dataset
# (We loaded 'dataset' back in Cell 3)
test_batch = dataset.select(range(1000))

print("=== 🚀 Starting 1,000-Question Stress Test ===")
print("Warning: This will take 20-30 minutes on a free GPU!\n")

safe_count = 0
blocked_count = 0

# 3. Loop through all 1000 questions
for data in tqdm(test_batch, desc="Generating & Validating SQL"):
    # Extract the question and schema from the dataset
    question = data['question']
    
    # Spider uses 'query' for the SQL and 'db_id' for the database name. 
    # We will use db_id as a simple schema placeholder for this test.
    schema = data['db_id'] 
    
    # Format the prompt exactly how the AI was trained
    prompt = f"Question: {question}\nSchema: {schema}\nSQL: "
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Generate the AI's answer
    # We keep max_new_tokens at 64 to force it to be relatively fast
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    # Clean the output to just get the AI's generated SQL
    try:
        # Split the text to isolate what comes after "SQL: "
        ai_sql = generated_text.split("SQL: ")[1].strip()
    except IndexError:
        # If the AI hallucinates and messes up the format, grab the whole thing
        ai_sql = generated_text 
    
    # 4. Pass the AI's SQL through YOUR Gatekeeper
    is_valid, message = validate_sql(ai_sql)
    
    if is_valid:
        safe_count += 1
    else:
        blocked_count += 1

# 5. Print the Final Analytics Report
print("\n\n=== ✨ Stress Test Analytics ✨ ===")
print(f"Total Queries Processed: 1000")
print(f"✅ Approved by Gatekeeper: {safe_count}")
print(f"🚫 Blocked by Gatekeeper:  {blocked_count}")
print("==================================")

=== 🚀 Starting 1,000-Question Stress Test ===



Generating & Validating SQL:   0%|          | 0/1000 [00:00<?, ?it/s]Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings



=== ✨ Stress Test Analytics ✨ ===
Total Queries Processed: 1000
✅ Approved by Gatekeeper: 1000
🚫 Blocked by Gatekeeper:  0


In [31]:
# ==========================================
# 1. SAVE LOCALLY (To Kaggle Output Files)
# ==========================================
# This saves the LoRA adapters to a folder named 'sql_agent_model'
model.save_pretrained("sql_agent_model")
tokenizer.save_pretrained("sql_agent_model")

print("✅ Model successfully saved locally to 'sql_agent_model' folder!")

# ==========================================
# 2. PUSH TO CLOUD (Hugging Face Hub) - Optional but Recommended
# ==========================================
# If you have a free Hugging Face account, you can back your model up to the cloud.
# Remove the '#' from the lines below and replace with your details to use this:

hf_username = "Ilyankhan69"
hf_token = "Your HF Token here" # Get this from huggingface.co/settings/tokens
repo_name = f"{hf_username}/schema-aware-sql-agent"

print(f"Pushing model to Hub: {repo_name}...")
model.push_to_hub(repo_name, token=hf_token)
tokenizer.push_to_hub(repo_name, token=hf_token)
print("✅ Model successfully pushed to Hugging Face Hub!")

Unsloth: Restored added_tokens_decoder metadata in sql_agent_model/tokenizer_config.json.


✅ Model successfully saved locally to 'sql_agent_model' folder!
Pushing model to Hub: Ilyankhan69/schema-aware-sql-agent...


README.md:   0%|          | 0.00/547 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Ilyankhan69/schema-aware-sql-agent


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpzx5kvbqg/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Model successfully pushed to Hugging Face Hub!
